In [1]:
!pip install -q torch torchaudio librosa soundfile datasets torchcodec
!pip install -q huggingface_hub
!pip install -q evaluate jiwer

("IMPORTANT INSTALLATION DONE")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.2 MB/s eta 0:00:00


'IMPORTANT INSTALLATION DONE'

In [18]:
import os
import random
import numpy as np
import pandas as pd
import torch
import json
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import librosa
import librosa.display
from datasets import load_dataset, Audio
import torchcodec
from jiwer import wer, cer
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from IPython.display import Audio, display
from IPython.display import Audio as IPyAudio, display
import sentencepiece as spm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

print("IMPORTING DONE")

IMPORTING DONE


In [62]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load mapping dictionaries
with open("/content/drive/MyDrive/VOICE2ENGLISH/num_to_char.json", "r") as f:
    num_to_char = json.load(f)
num_to_char = {int(k): v for k, v in num_to_char.items()}

with open("/content/drive/MyDrive/VOICE2ENGLISH/char_to_num.json", "r") as f:
    char_to_num = json.load(f)
char_to_num = {k: int(v) for k, v in char_to_num.items()}



In [ ]:
dataset = load_dataset("Purvaxxx/hindi_test_dataset")
print(dataset)

In [6]:
spectrogram_transform = T.Spectrogram(
    n_fft=256,
    hop_length=160,
    win_length=256,
    power=2.0
).to(device)

def preprocess_audio(audio_dict):

    audio_array = audio_dict["array"]
    sr = audio_dict["sampling_rate"]

    # Convert to tensor (1D waveform)
    waveform = torch.tensor(audio_array, dtype=torch.float32, device=device)
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)  # (1, time)

    # Compute power spectrogram
    spectrogram = spectrogram_transform(waveform)  # (1, freq, time)

    # Convert to magnitude spectrogram (sqrt of power)
    spectrogram = torch.sqrt(spectrogram + 1e-10)

    # Permute to (time, freq) for model
    spectrogram = spectrogram.squeeze(0).permute(1, 0)

    # Normalize per-frequency
    means = spectrogram.mean(dim=0, keepdim=True)
    stddevs = spectrogram.std(dim=0, keepdim=True)
    spectrogram = (spectrogram - means) / (stddevs + 1e-10)

    return spectrogram.cpu().numpy()


In [7]:
class PositionalEncodingNMT(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (seq_len, batch, d_model)
        return x + self.pe[:x.size(0)]

In [8]:
class ASRModel(nn.Module):
    def __init__(self, input_dim, output_dim, rnn_layers=2, rnn_units=128, conv_channels=16):
        super().__init__()

        # Define kernel sizes and strides
        kernel_size1 = (11, 41)
        stride1 = (2, 2)
        kernel_size2 = (11, 21)
        stride2 = (1, 2)  # This stride does not affect the time dimension

        # Calculate padding for Conv1 and Conv2
        padding1 = ((kernel_size1[0] - 1) // 2, (kernel_size1[1] - 1) // 2)
        padding2 = ((kernel_size2[0] - 1) // 2, (kernel_size2[1] - 1) // 2)

        # Convolutional layers
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, conv_channels, kernel_size=kernel_size1, stride=stride1, padding=padding1, bias=False),
            nn.BatchNorm2d(conv_channels),
            nn.ReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(conv_channels, conv_channels, kernel_size=kernel_size2, stride=stride2, padding=padding2, bias=False),
            nn.BatchNorm2d(conv_channels),
            nn.ReLU()
        )

        # Calculate RNN input size after convolutions
        with torch.no_grad():
            dummy_input = torch.randn(1, 1, 100, input_dim)  # (batch, channels, time, features)
            dummy_output = self.conv1(dummy_input)
            dummy_output = self.conv2(dummy_output)
            rnn_input_features = dummy_output.shape[1] * dummy_output.shape[3]  # channels * freq_dim

        # Stacked BiLSTMs
        self.rnns = nn.ModuleList()
        for i in range(rnn_layers):
            self.rnns.append(
                nn.Sequential(
                    nn.LSTM(
                        input_size=rnn_input_features if i == 0 else rnn_units * 2,
                        hidden_size=rnn_units,
                        bidirectional=True,
                        batch_first=True
                    ),
                    nn.Dropout(0.5) if i < rnn_layers - 1 else nn.Identity()
                )
            )

        # Dense + Classifier
        self.dense1 = nn.Sequential(
            nn.Linear(rnn_units * 2, rnn_units),  # 256 → 128
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.classifier = nn.Linear(rnn_units, output_dim)  # 128 → vocab_size

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv1(x)
        x = self.conv2(x)


        batch_size, channels, time, features = x.shape
        x = x.permute(0, 2, 1, 3).contiguous().view(batch_size, time, channels * features)

        for i, rnn_layer in enumerate(self.rnns):
            x, _ = rnn_layer[0](x)  # LSTM output
            if isinstance(rnn_layer[1], nn.Dropout):
                x = rnn_layer[1](x)

        x = self.dense1(x)
        x = self.classifier(x)
        return x

print("ASR ARCHITECTURE DEFINED")
PAD_IDX = 0
class NMTTransformer(nn.Module):
    def __init__(self,
                 src_vocab_size,
                 tgt_vocab_size,
                 d_model,
                 nhead,
                 num_layers,
                 dim_feedforward,
                 dropout
                ):
        super().__init__()
        self.src_embed = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_enc = PositionalEncodingNMT(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):

        # Masks
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(src.device)
        src_key_padding_mask = (src == PAD_IDX)
        tgt_key_padding_mask = (tgt == PAD_IDX)

        # Embeddings and positional encoding
        # format(batch, seq_len, d_model)
        src_emb = self.pos_enc(self.src_embed(src))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt))

        # Transformer - (seq_len, batch, d_model)
        src_emb = src_emb.transpose(0, 1)
        tgt_emb = tgt_emb.transpose(0, 1)

        output = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )

        return self.fc_out(output.transpose(0, 1))


print("NMT ARCHITECTURE DEFINED")


ASR ARCHITECTURE DEFINED
NMT ARCHITECTURE DEFINED


In [9]:
#CONSTANTS
input_dim = 256 // 2 + 1
output_dim = len(char_to_num)
rnn_units = 128
rnn_layers = 2
conv_channels = 32

device = "cuda" if torch.cuda.is_available() else "cpu"
asr_model = ASRModel(input_dim,
                 output_dim,
                 rnn_layers=rnn_layers,
                 rnn_units=rnn_units,
                 conv_channels=conv_channels).to(device)

nmt_model = NMTTransformer(
    src_vocab_size = 4000,
    tgt_vocab_size = 4000,
    d_model=512,
    nhead=4,
    num_layers=6,
    dim_feedforward=512,
    dropout=0.1
).to(device)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [10]:
# Load ASR model checkpoint
CTC_ASR = "/content/drive/MyDrive/VOICE2ENGLISH/CTC_asr_epoch26.pt"
asr_checkpoint = torch.load(CTC_ASR, map_location=device)

asr_model.load_state_dict(asr_checkpoint["model_state_dict"])
asr_model.to(device)
asr_model.eval()

print("ASR model loaded")

Transformer_NMT = "/content/drive/MyDrive/VOICE2ENGLISH/checkpoint_epoch_60 (1).pt"
nmt_checkpoint = torch.load(Transformer_NMT, map_location=device)
nmt_model.load_state_dict(nmt_checkpoint["model_state_dict"])
nmt_model.to(device)
nmt_model.eval()
print("NMT model loaded")

spm_src = "/content/drive/MyDrive/VOICE2ENGLISH/spm_src.model"
spm_tgt = "/content/drive/MyDrive/VOICE2ENGLISH/spm_tgt.model"
sp_src = spm.SentencePieceProcessor(model_file=spm_src)
sp_tgt = spm.SentencePieceProcessor(model_file=spm_tgt)
print("SentencePiece models loaded.")



ASR model loaded
NMT model loaded
SentencePiece models loaded.


In [11]:
pad_id = 0
sos_id = 1
eos_id = 2

# DECODING
def greedy_decode(model, src, max_len, device):
    model.eval()
    src = src.to(device)

    src_key_padding_mask = (src == 0)

    src_emb = model.pos_enc(model.src_embed(src)).transpose(0, 1)
    memory = model.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

    ys = torch.tensor([[sos_id]], dtype=torch.long, device=device)

    for _ in range(max_len):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(ys.size(1)).to(device)
        tgt_emb = model.pos_enc(model.tgt_embed(ys)).transpose(0, 1)

        out = model.transformer.decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            memory_key_padding_mask=src_key_padding_mask)

        out = model.fc_out(out.transpose(0, 1))
        next_token = out[:, -1, :].argmax(dim=-1).item()
        ys = torch.cat([ys, torch.tensor([[next_token]], device=device)], dim=1)
        if next_token == eos_id:
            break
    return ys.squeeze(0).tolist()

print("DONE")

DONE


In [12]:
def decode_ctc_prediction(output_logits, num_to_char, blank_idx=0):
    pred_ids = torch.argmax(output_logits, dim=-1).cpu().numpy().tolist()

    decoded_tokens = []
    prev_token = None
    for token_id in pred_ids:
        if token_id != blank_idx and token_id != prev_token:
            decoded_tokens.append(token_id)
        prev_token = token_id

    return "".join([num_to_char[t] for t in decoded_tokens])

In [13]:
def load_and_preprocess(audio_dict):
    audio_array = audio_dict["array"]
    sr = audio_dict["sampling_rate"]
    display(IPyAudio(audio_array, rate=sr))
    spec = preprocess_audio(audio_dict)
    return spec

In [14]:
def predict_from_mel(spec_np, model, device, max_len=150):
    spec = torch.from_numpy(spec_np).float().unsqueeze(0).to(device)
    with torch.no_grad():
        output_logits = model(mel_tensor)
    pred_text = decode_ctc_prediction(output_logits[0], num_to_char)
    return pred_text

In [15]:
def translate_text(text, nmt_model, sp_src, sp_tgt, device, max_len=150):

    src_ids = sp_src.encode(text, out_type=int)
    src_tensor = torch.tensor([src_ids], device=device)
    pred_ids = greedy_decode(nmt_model, src_tensor, max_len=max_len, device=device)

    vocab_size = sp_tgt.get_piece_size()
    pred_ids_filtered = [i for i in pred_ids if i not in [pad_id,sos_id,eos_id] and 0 <= i < vocab_size]

    translated_text = sp_tgt.decode(pred_ids_filtered)
    return translated_text

In [16]:
def speech_to_translation(mel_np, asr_model, nmt_model, sp_src, sp_tgt, device):
    asr_text = predict_from_mel(mel_np, asr_model, device)
    translated_text = translate_text(asr_text, nmt_model, sp_src, sp_tgt, device)
    return asr_text, translated_text


In [57]:
#TESTING
idx = random.randint(0, len(dataset["hindi"]) - 1)
sample = dataset["hindi"][idx]

mel = load_and_preprocess(sample["chunked_audio_filepath"])

mel_tensor = torch.from_numpy(mel).float().unsqueeze(0).to(device)

ASR_text, NMT_text = speech_to_translation(
    mel, asr_model, nmt_model, sp_src, sp_tgt, device
)

print(f"Hindi : {sample['text']}")
print(f"English : {sample['en_text']}")
print("\n")
print("OUR MODEL OUTPUT")
print(f"Source Text = : {ASR_text}")
print(f"English Text = : {NMT_text}")

ground_asr = sample["text"]
asr_output = ASR_text
ground_en = sample["en_text"]
nmt_output = NMT_text

asr_wer = wer(ground_asr, asr_output)
nmt_bleu = sentence_bleu(
    [ground_en.split()],
    nmt_output.split(),
    smoothing_function=SmoothingFunction().method1
)
print(f"ASR WER: {asr_wer:.4f}")
print(f"NMT BLEU Score: {nmt_bleu:.4f}")

Hindi : उस ने उसके उत्तर में उन से कहा कि मेरी माता और मेरे भाई ये ही हैं जो परमेश्वर का वचन सुनते और मानते हैं
English : And he answered and said unto them, My mother and my brethren are these which hear the word of God, and do it


OUR MODEL OUTPUT
Source Text = : उस ने उसके उत्तर में उन से कहा कि मेरी माता और मेरे धाई योही है जो परमेश्वर का वचन सुनते और मानते हैं
English Text = : and he answered and said unto them, behold, my mother and my brethren are these which hear the word of god, and do nothing of him
ASR WER: 0.1600
NMT BLEU Score: 0.5900


In [65]:
from IPython.display import Audio as IPyAudio, display
audio_sample1 = "/content/Sample05.wav"
array, sr = librosa.load(audio_sample1, sr=16000)

display(IPyAudio(array, rate=sr))

In [67]:
audio_dict = {
    "array": array,
    "sampling_rate": sr
}

mel = preprocess_audio(audio_dict)
mel_tensor = torch.tensor(mel).unsqueeze(0).float().to(device)
with torch.no_grad():
    output_logits = asr_model(mel_tensor)
ASR_text = decode_ctc_prediction(output_logits[0], num_to_char)
print(f" Source Text: {ASR_text}")
nmt_text = translate_text(ASR_text, nmt_model, sp_src, sp_tgt, device)
print(f" English Text: {nmt_text}")

 Source Text: हारीम की सन्तान तीन सौ बीस
 English Text: the children of harim, three hundred and twenty
